In [ ]:
# =========================
# 0. MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount("/content/drive")

# =========================
# 1. INSTALL DEPENDENCIES
# =========================
!pip install -U transformers accelerate pillow matplotlib

# =========================
# 2. IMPORT LIBRARIES
# =========================
import os, json, re, torch
from PIL import Image, ImageDraw
from transformers import AutoProcessor, AutoModelForVision2Seq

# =========================
# 3. PATHS (MATCH YOUR FOLDER)
# =========================
INPUT_IMAGE = "/content/drive/MyDrive/qwen_images/confined_space.png"
OUTPUT_DIR = "/content/drive/MyDrive/qwen_images/results4-confined_space"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 4. LOAD QWEN-VL MODEL
# =========================
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLUAc

In [ ]:
HAZARD_TEXT = """
You are a construction safety inspector.

This image shows a confined space entry and rescue operation.

Task:
- Identify a CONFINED SPACE hazard.
- Focus on limited entry/exit, oxygen deficiency risk,
  and reliance on retrieval/rescue systems.
- The PRIMARY hazard is restricted space with potential
  atmospheric danger.

Rules:
- Always classify this scene as "Confined space and oxygen deficiency".
- Do NOT classify as caught-in/between or vehicle accident.
- Draw ONE bounding box around the confined space opening
  (manhole or shaft), not the worker.

Return ONLY a single valid JSON object.
Begin with { and end with }.

{
  "hazard_family": "Confined space and oxygen deficiency",
  "hazard_category": "Confined space entry / rescue operation",
  "bbox_xyxy": [x1, y1, x2, y2],
  "confidence": 0.0,
  "reason": ""
}


"""

# =========================
# 6. ROBUST JSON EXTRACTION
# =========================
def extract_json(response):
    response = re.sub(r"```json|```", "", response, flags=re.IGNORECASE)
    matches = re.findall(r"\{[\s\S]*?\}", response)
    if not matches:
        raise ValueError("No JSON object found")
    return json.loads(matches[-1])

# =========================
# 7. RUN QWEN ON IMAGE
# =========================
def run_qwen_on_image(image_path):
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": HAZARD_TEXT}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    response = processor.batch_decode(output, skip_special_tokens=True)[0]
    return image, response

# =========================
# 8. DRAW BOUNDING BOX
# =========================
def draw_bbox(image, result, save_path):
    draw = ImageDraw.Draw(image)
    x1, y1, x2, y2 = result["bbox_xyxy"]
    draw.rectangle([x1, y1, x2, y2], outline="red", width=4)
    draw.text((x1, max(0, y1 - 15)), result["hazard_family"], fill="red")
    image.save(save_path)

# =========================
# 9. PROCESS SINGLE IMAGE
# =========================
print("🔍 Processing vehicle accident image")

image, response = run_qwen_on_image(INPUT_IMAGE)
result = extract_json(response)

boxed_path = os.path.join(OUTPUT_DIR, "boxed_heavy.png")
draw_bbox(image, result, boxed_path)

with open(os.path.join(OUTPUT_DIR, "heavy.json"), "w") as f:
    json.dump(result, f, indent=2)

print("✅ DONE →", result["hazard_family"])

🔍 Processing vehicle accident image
✅ DONE → Confined space and oxygen deficiency
